# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [2]:
ПРІЗВИЩЕ_ІМЯ = 'Ореньчук Юрій'   # прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 15                # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [3]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)


ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 15   ·   Ореньчук Юрій   ·   ШІ-31

Ваші вісім стовпців:
   1. Телефон
   2. Стать
   3. ступінь КТ
   4. Виписка
   5. КТ
   6. Т вище 37,5
   7. Сечов2
   8. БРпр1

Ваша пара для аналізу зв’язку: Стать та ступінь КТ

Ваше додаткове питання:
   Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?


In [4]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Стать,ступінь КТ,Виписка,КТ,"Т вище 37,5",Сечов2,БРпр1
0,0958469788 дружина Ганна,ч,сердній,2022-03-10 00:00:00,20-30%,3,6.5,3.4
1,"0671778486, 0994887663 дружина Оксана",ч,середній,2022-03-16 00:00:00,30-35%,3,4,2.8
2,"0958768045, чоловік Степан 0951784726",ж,критичний,2022-01-18 00:00:00,70-80%,3,10.1,2.8
3,0959819331. дочка Валентина 0960845205,ж,критичний,2022-01-13 00:00:00,0.8,6,12.5,3.4
4,"0952294866, дочка Дарина 0678360161",ж,NaN,2022-03-11 00:00:00,NaN,5 до гос,7.4,3.4
5,0959922983 сестра Софія,ж,середній,2021-12-26 00:00:00,30-40%,NaN,21.4,3.4
6,"0998073702, дружина Наталія 0998382226",ч,важкий,2022-05-11 00:00:00,60-65%,NaN,6.1,5.1
7,"0637258771, син Сергій 0968413301",ж,легкий,2022-05-28 00:00:00,до 17%,NaN,7.6,5.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [5]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Телефон      120 non-null    object 
 1   Стать        121 non-null    str    
 2   ступінь КТ   42 non-null     str    
 3   Виписка      120 non-null    object 
 4   КТ           44 non-null     object 
 5   Т вище 37,5  78 non-null     object 
 6   Сечов2       102 non-null    object 
 7   БРпр1        109 non-null    float64
dtypes: float64(1), object(5), str(2)
memory usage: 7.7+ KB


In [6]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Стать
    непорожніх 121 з 121 | унікальних 4 | типи: str×121
    приклади: ["'ч'", "'ж'", "'чол'", "'жін'"]

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5
    непорожніх 78 з 121 | унікальних 35 | типи: str×48, int×30
    приклади: ['3', '6', "

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [7]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор',
        'Номер телефону родича це мітка для зв’язку, а не величина; у клітинках ще й імена родичів та по кілька номерів.'),
    'Стать': ('номінальна',
        'Дві категорії (ч/ж) можна лише порівняти на рівність, порядку між ними немає; «ч» і «чол» це той самий рівень.'),
    'ступінь КТ': ('порядкова',
        'Ступені (легкий < середній < важкий < критичний) мають природний порядок, але відстані між сусідніми ступенями невідомі.'),
    'Виписка': ('інтервальна',
        'Це дата: різниця дат у днях має сенс, але нуль календаря умовний, тому відношення дат («удвічі пізніше») безглузді.'),
    'КТ': ('відношень',
        'Частка ураженої легеневої тканини за КТ: 0 % справжня відсутність ураження, і 60 % справді вдвічі більше за 30 %.'),
    'Т вище 37,5': ('абсолютна',
        'Лічильник днів із температурою вище 37,5: є природний нуль (жодного дня) і природна одиниця це один день.'),
    'Сечов2': ('відношень',
        'Концентрація сечовини в крові при повторному аналізі: природний нуль, відношення значень мають сенс.'),
    'БРпр1': ('відношень',
        'Концентрація прямого білірубіну: природний нуль, відношення мають сенс; значення кратні 1,7 це крок вимірювання грубий.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,"Номер телефону родича це мітка для зв’язку, а не величин..."
1,Стать,номінальна,"Дві категорії (ч/ж) можна лише порівняти на рівність, по..."
2,ступінь КТ,порядкова,Ступені (легкий < середній < важкий < критичний) мають п...
3,Виписка,інтервальна,"Це дата: різниця дат у днях має сенс, але нуль календаря..."
4,КТ,відношень,Частка ураженої легеневої тканини за КТ: 0 % справжня ві...
5,"Т вище 37,5",абсолютна,"Лічильник днів із температурою вище 37,5: є природний ну..."
6,Сечов2,відношень,Концентрація сечовини в крові при повторному аналізі: пр...
7,БРпр1,відношень,"Концентрація прямого білірубіну: природний нуль, відноше..."


In [8]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.



### Очищення значень (потрібне для етапів 2–5)

Дані заповнювали вручну, тому в одному стовпці живуть різні написання одного й того ж.
Перш ніж рахувати статистики, зводимо їх до єдиного вигляду. Кожне перетворення — окрема
функція, щоб було видно, що саме ми припустили.

| Стовпець | Що знайдено | Як чистимо |
|---|---|---|
| Стать | `ч`, `чол`, `ж`, `жін` | два рівні: `ч` / `ж` |
| ступінь КТ | 12 написань, помилки («сердній», «серед ступ»), зайві пробіли | 4 впорядковані рівні |
| Виписка | 118 дат і 2 рядки-помилки (`24.024.2021`, `20.24.2021`) | `NaT` для нерозпізнаних |
| КТ | частки `0.3`, відсотки `30%`, діапазони `30-35%`, `до 17%`, порожній рядок | частка 0–1, діапазон → середина |
| Т вище 37,5 | числа й текст («3 дні в стац», «5 днів до госп») | сума чисел у рядку |
| Сечов2 | числа і слово «немає» | `NaN` |

**Припущення щодо «вкрай важкий» і «критичний».** Це два написання одного верхнього ступеня:
у рядках з обома мітками відсоток ураження за КТ однаковий (70–80 %). Тому ступенів чотири,
як у класифікації КТ-1…КТ-4.


In [9]:

import re
from scipy import stats

ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'критичний']   # від меншого до більшого


def чист_стать(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    return 'ч' if s.startswith('ч') else 'ж' if s.startswith('ж') else np.nan


def чист_ступінь(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s.startswith(('вкрай', 'критич')):
        return 'критичний'
    if s.startswith('важк'):
        return 'важкий'
    if s.startswith(('серд', 'серед')):
        return 'середній'
    if s.startswith('легк'):
        return 'легкий'
    return np.nan


def чист_КТ(x):
    """Частка ураження 0–1. Діапазон замінюємо серединою, «до 17%» — верхньою межею."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip().replace('%', '').replace(' ', '').replace(',', '.')
    if not s:
        return np.nan
    if s.startswith('до'):
        return float(s[2:]) / 100
    if '-' in s:
        a, b = s.split('-')
        return (float(a) + float(b)) / 2 / 100
    v = float(s)
    return v / 100 if v > 1 else v


def чист_дні(x):
    """Загальна кількість днів із температурою: сума всіх чисел у рядку."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    числа = re.findall(r'\d+', str(x))
    return float(sum(map(int, числа))) if числа else np.nan


ОЧИЩЕННЯ = {
    'Стать':       lambda s: s.map(чист_стать),
    'ступінь КТ':  lambda s: s.map(чист_ступінь),
    'Виписка':     lambda s: pd.to_datetime(s, errors='coerce'),
    'КТ':          lambda s: s.map(чист_КТ),
    'Т вище 37,5': lambda s: s.map(чист_дні),
    'Сечов2':      lambda s: pd.to_numeric(s, errors='coerce'),
    'БРпр1':       lambda s: pd.to_numeric(s, errors='coerce'),
    'Телефон':     lambda s: s.astype(str).where(s.notna()),
}


def очищений(стовпець):
    return ОЧИЩЕННЯ[стовпець](df[стовпець])


print('Стать після очищення:', очищений('Стать').value_counts().to_dict())
print('ступінь КТ після очищення:',
      очищений('ступінь КТ').value_counts().reindex(ПОРЯДОК_КТ).to_dict())
print('Виписка: нерозпізнаних дат: ', int(очищений('Виписка').isna().sum()) - int(df['Виписка'].isna().sum()))


Стать після очищення: {'ж': 75, 'ч': 46}
ступінь КТ після очищення: {'легкий': 7, 'середній': 18, 'важкий': 10, 'критичний': 7}
Виписка: нерозпізнаних дат:  2


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [10]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = очищений(стовпець).dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш == 'ідентифікатор':
        # усі значення унікальні, тому мода й частоти не несуть інформації
        р['усі значення унікальні'] = bool(з.nunique() == len(з))
        return р

    # номінальна — і мода з частотами для всіх інших шкал
    if ш == 'номінальна':
        р['мода'] = з.mode().iloc[0]
        р['частоти'] = з.value_counts().to_dict()
        return р

    if ш == 'порядкова':
        # працюємо з кодами впорядкованої категорії, щоб медіана була реальним рівнем
        кат = pd.Categorical(з, categories=ПОРЯДОК_КТ, ordered=True)
        коди = pd.Series(кат.codes)
        р['мода'] = з.mode().iloc[0]
        р['частоти'] = з.value_counts().reindex(ПОРЯДОК_КТ).to_dict()
        р['медіана'] = ПОРЯДОК_КТ[int(np.quantile(коди, 0.5, method='lower'))]
        р['квантиль 25%'] = ПОРЯДОК_КТ[int(np.quantile(коди, 0.25, method='lower'))]
        р['квантиль 75%'] = ПОРЯДОК_КТ[int(np.quantile(коди, 0.75, method='higher'))]
        return р

    if ш == 'інтервальна':
        # для дат: середня дата й розкид у днях; відношень дат не обчислюємо
        р['мінімум'] = з.min().date()
        р['медіана'] = з.sort_values().iloc[len(з) // 2].date()
        р['середня дата'] = з.mean().date()
        р['ст. відхилення, днів'] = round(з.std().total_seconds() / 86400, 1)
        р['максимум'] = з.max().date()
        return р

    if ш in ('відношень', 'абсолютна'):
        р['мода'] = з.mode().iloc[0]
        р['мінімум'] = з.min()
        р['q25'], р['медіана'], р['q75'] = з.quantile([0.25, 0.5, 0.75]).round(3).tolist()
        р['максимум'] = з.max()
        р['середнє'] = round(з.mean(), 3)
        р['ст. відхилення'] = round(з.std(), 3)
        р['коефіцієнт варіації'] = round(з.std() / з.mean(), 3)
        return р

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120, 'усі значення унікальні': True}

--- Стать  (номінальна)
    {'непорожніх': 121, 'унікальних': 2, 'мода': 'ж', 'частоти': {'ж': 75, 'ч': 46}}

--- ступінь КТ  (порядкова)
    {'непорожніх': 42, 'унікальних': 4, 'мода': 'середній', 'частоти': {'легкий': 7, 'середній': 18, 'важкий': 10, 'критичний': 7}, 'медіана': 'середній', 'квантиль 25%': 'середній', 'квантиль 75%': 'важкий'}

--- Виписка  (інтервальна)
    {'непорожніх': 118, 'унікальних': 88, 'мінімум': datetime.date(2021, 4, 27), 'медіана': datetime.date(2022, 3, 16), 'середня дата': datetime.date(2022, 1, 29), 'ст. відхилення, днів': 109.5, 'максимум': datetime.date(2022, 6, 22)}

--- КТ  (відношень)
    {'непорожніх': 43, 'унікальних': 21, 'мода': np.float64(0.3), 'мінімум': np.float64(0.05), 'q25': 0.262, 'медіана': 0.35, 'q75': 0.638, 'максимум': np.float64(0.8), 'середнє': np.float64(0.425), 'ст. відхилення': np.float64(0.22), 'коефіцієнт варіації': np.flo

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [11]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Стать,121,4,ж,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Т вище 37,5",78.0,35.0,3.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов2,102.0,66.0,5.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
БРпр1,109.0,NaN,NaN,NaN,4.801468,1.575191,2.4,3.4,5.1,5.3,10.2


In [12]:
тел_числа = pd.to_numeric(df['Телефон'], errors='coerce').dropna()
сер_тел = float(тел_числа.mean())

коди_кт = pd.Categorical(очищений('ступінь КТ').dropna(),
                         categories=ПОРЯДОК_КТ, ordered=True).codes + 1
сер_кт = float(коди_кт.mean())
сер_кт_інакше = float(np.array([1, 2, 3, 10])[коди_кт - 1].mean())

дати = очищений('Виписка').dropna()
серійні_1900 = (дати - pd.Timestamp('1899-12-30')).dt.days
серійні_1904 = (дати - pd.Timestamp('1904-01-01')).dt.days
кв_1900 = float(серійні_1900.std() / серійні_1900.mean())
кв_1904 = float(серійні_1904.std() / серійні_1904.mean())

відн_дат = float(серійні_1900.max() / серійні_1900.min())

БЕЗГЛУЗДІ = [
    ('Телефон', f'середнє ({len(тел_числа)} номерів, збережених як числа)', round(сер_тел),
     'Номер телефону це мітка, а не величина: «середній телефон» не відповідає жодному абоненту, '
     'а різниця номерів нічого не вимірює.'),
    ('ступінь КТ', 'середнє кодів 1–4', round(сер_кт, 2),
     f'Коди лише нумерують порядок. З кодами 1, 2, 3, 10 те саме середнє дорівнює {сер_кт_інакше:.2f}: '
     'результат залежить від довільного вибору чисел, тож «середній ступінь» не має змісту.'),
    ('Виписка', 'коефіцієнт варіації серійного номера дати', round(кв_1900, 5),
     f'Нуль дати умовний: з початком відліку 1904 року той самий показник дорівнює {кв_1904:.5f}. '
     'Коефіцієнт варіації потребує природного нуля, а в календаря його немає.'),
    ('Виписка', 'відношення макс. дати до мін. (серійні номери)', round(відн_дат, 4),
     'Відношення дат («у стільки разів пізніше») залежить від того, відкіля рахувати, і майже '
     'дорівнює 1 лише тому, що відлік почався 1900 року. Для інтервальної шкали змістовні тільки різниці.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,"середнє (23 номерів, збережених як числа)",7.587990e+08,"Номер телефону це мітка, а не величина: «середній телефо..."
1,ступінь КТ,середнє кодів 1–4,2.400000e+00,"Коди лише нумерують порядок. З кодами 1, 2, 3, 10 те сам..."
2,Виписка,коефіцієнт варіації серійного номера дати,2.460000e-03,Нуль дати умовний: з початком відліку 1904 року той сами...
3,Виписка,відношення макс. дати до мін. (серійні номери),1.009500e+00,Відношення дат («у стільки разів пізніше») залежить від ...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [13]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × ступінь КТ

Стать: шкала — номінальна
ступінь КТ: шкала — порядкова

  Стать ступінь КТ
0     ч    сердній
1     ч   середній
2     ж  критичний
3     ж  критичний
4     ж        NaN
5     ж   середній


In [ ]:
ОБРАНА_МІРА = 'коефіцієнт Крамера (V)'
ЧОМУ_САМЕ_ВОНА = ('Стать — номінальна, ступінь КТ — порядкова, а за таблицею вище для поєднання '
                  'номінальна × порядкова беруть V Крамера: він не припускає ні порядку, ні рівних відстаней між категоріями.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

# --- обчислення ---
пара = pd.DataFrame({'Стать': очищений('Стать'),
                     'ступінь КТ': очищений('ступінь КТ')}).dropna()
таблиця = pd.crosstab(пара['Стать'], пара['ступінь КТ']).reindex(columns=ПОРЯДОК_КТ)
print('Таблиця спряженості (рядки, у яких відомі обидві змінні):')
print(таблиця)
print()

n = int(таблиця.values.sum())
χ2, p_χ2, dof, очікувані = stats.chi2_contingency(таблиця, correction=False)
k, r = таблиця.shape[1], таблиця.shape[0]
V = float(np.sqrt(χ2 / (n * (min(k, r) - 1))))

# поправка на зміщення для малих вибірок (Bergsma, 2013)
φ2 = χ2 / n
φ2_поправлений = max(0, φ2 - (k - 1) * (r - 1) / (n - 1))
k_поправлене = k - (k - 1) ** 2 / (n - 1)
r_поправлене = r - (r - 1) ** 2 / (n - 1)
V_поправлений = float(np.sqrt(φ2_поправлений / (min(k_поправлене, r_поправлене) - 1)))

частка_малих = float((очікувані < 5).mean())

# p-значення перестановками: χ²-наближення ненадійне, коли очікувані частоти < 5
rng = np.random.default_rng(15)
стать_значення = пара['Стать'].to_numpy()
ступінь_значення = пара['ступінь КТ'].to_numpy()
χ2_перестановок = []
for _ in range(5000):
    t = pd.crosstab(rng.permutation(стать_значення), ступінь_значення)
    χ2_перестановок.append(stats.chi2_contingency(t, correction=False)[0])
p_перестановок = float((np.array(χ2_перестановок) >= χ2 - 1e-12).mean())

print(f'n = {n}, χ² = {χ2:.3f}, ступенів свободи = {dof}, p (χ²) = {p_χ2:.3f}')
print(f'V Крамера = {V:.3f}; з поправкою на малу вибірку = {V_поправлений:.3f}')
print(f'p за перестановками (5000) = {p_перестановок:.3f}')
print(f'Очікувані частоти < 5 у {частка_малих:.0%} клітинок — χ² лише орієнтовний.')

Таблиця спряженості (рядки, у яких відомі обидві змінні):
ступінь КТ  легкий  середній  важкий  критичний
Стать                                          
ж                4         8       4          5
ч                3        10       6          2

n = 42, χ² = 2.051, ступенів свободи = 3, p (χ²) = 0.562
V Крамера = 0.221; з поправкою на малу вибірку = 0.000
p за перестановками (5000) = 0.617
Очікувані частоти < 5 у 50% клітинок: χ² лише орієнтовний.


In [ ]:
# Контрприклад: Пірсон на числових кодах номінальної та порядкової змінних.
# Кодуємо категорії числами трьома різними, однаково «законними» способами.
пара['ступінь_код'] = pd.Categorical(пара['ступінь КТ'], categories=ПОРЯДОК_КТ,
                                     ordered=True).codes + 1

кодування_статі = {
    'ч=1, ж=2':  {'ч': 1, 'ж': 2},
    'ч=2, ж=1':  {'ч': 2, 'ж': 1},
    'ч=0, ж=10': {'ч': 0, 'ж': 10},
}
кодування_ступеня = {
    'коди 1,2,3,4':  {1: 1, 2: 2, 3: 3, 4: 4},
    'коди 1,2,3,10': {1: 1, 2: 2, 3: 3, 4: 10},
    'коди 4,3,2,1':  {1: 4, 2: 3, 3: 2, 4: 1},
}

рядки = []
for н_с, к_с in кодування_статі.items():
    for н_к, к_к in кодування_ступеня.items():
        x = пара['Стать'].map(к_с)
        y = пара['ступінь_код'].map(к_к)
        рядки.append((н_с, н_к, round(float(np.corrcoef(x, y)[0, 1]), 3)))

пірсон_таблиця = (pd.DataFrame(рядки, columns=['кодування статі', 'кодування ступеня', 'r Пірсона'])
                  .pivot(index='кодування статі', columns='кодування ступеня', values='r Пірсона'))
print(пірсон_таблиця)
print()
print(f'Для порівняння, V Крамера не залежить від кодування: {V:.3f}')
print()
print('Висновок: r змінює знак і величину від того, якими числами ми позначили категорії,')
print('хоча самі дані не змінилися. Число, яке залежить від довільного кодування, не можна інтерпретувати.')

кодування ступеня  коди 1,2,3,10  коди 1,2,3,4  коди 4,3,2,1
кодування статі                                             
ч=0, ж=10                  0.166         0.075        -0.075
ч=1, ж=2                   0.166         0.075        -0.075
ч=2, ж=1                  -0.166        -0.075         0.075

Для порівняння, V Крамера не залежить від кодування: 0.221

Висновок: r змінює знак і величину від того, якими числами ми позначили категорії,
хоча самі дані не змінилися. Число, яке залежить від довільного кодування, не можна інтерпретувати.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [16]:
ПІДГОТОВКА = {
    'Телефон':     'У модель не брати взагалі: ідентифікатор, а не величина.',
    'Стать':       'One-hot кодування після зведення написань до двох рівнів; порядок категорій не вводимо.',
    'ступінь КТ':  'Цілі коди 0–3 із заданим порядком легкий < середній < важкий < критичний; пропуски лишаємо NaN.',
    'Виписка':     'Перейти від дати до тривалості: Виписка − Поступлення в днях, далі стандартизація.',
    'КТ':          'Звести до частки 0–1 (діапазон -> середина), далі стандартизація; пропусків багато, тож без вигаданих значень.',
    'Т вище 37,5': 'Витягти число днів із тексту; лічильник стандартизувати, за скошеності log1p.',
    'Сечов2':      'Слово «немає» замінити на NaN, далі стандартизація; логарифм, якщо розподіл скошений вправо.',
    'БРпр1':       'Логарифмування log1p через правий хвіст і стандартизація; пам’ятати про дискретність значень.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"У модель не брати взагалі: ідентифікатор, а не величина."
1,Стать,номінальна,One-hot кодування після зведення написань до двох рівнів...
2,ступінь КТ,порядкова,Цілі коди 0–3 із заданим порядком легкий < середній < ва...
3,Виписка,інтервальна,Перейти від дати до тривалості: Виписка − Поступлення в ...
4,КТ,відношень,"Звести до частки 0–1 (діапазон -> середина), далі станда..."
5,"Т вище 37,5",абсолютна,"Витягти число днів із тексту; лічильник стандартизувати,..."
6,Сечов2,відношень,"Слово «немає» замінити на NaN, далі стандартизація; лога..."
7,БРпр1,відношень,Логарифмування log1p через правий хвіст і стандартизація...


In [17]:
X = pd.DataFrame(index=df.index)

# номінальна: Стать -> one-hot
X = X.join(pd.get_dummies(очищений('Стать'), prefix='Стать').astype(int))

# порядкова: ступінь КТ -> цілі коди
кат = pd.Categorical(очищений('ступінь КТ'), categories=ПОРЯДОК_КТ, ordered=True)
X['ступінь_КТ_код'] = pd.Series(кат.codes, index=df.index).replace(-1, np.nan)

# інтервальна: дата -> тривалість у днях
тривалість = (очищений('Виписка') - pd.to_datetime(df['Поступлення'], errors='coerce')).dt.days
if тривалість.std() > 0:
    X['тривалість_днів_z'] = (тривалість - тривалість.mean()) / тривалість.std()

# відношень: БРпр1 -> log1p -> стандартизація
лог = np.log1p(очищений('БРпр1'))
if лог.std() > 0:
    X['БРпр1_log_z'] = (лог - лог.mean()) / лог.std()

print('Форма підготовленої матриці:', X.shape)
print('Пропусків по стовпцях:'); print(X.isna().sum().to_dict())
print()
print(X.describe().loc[['count', 'mean', 'std', 'min', 'max']].round(3).T)

Форма підготовленої матриці: (121, 5)
Пропусків по стовпцях:
{'Стать_ж': 0, 'Стать_ч': 0, 'ступінь_КТ_код': 79, 'тривалість_днів_z': 3, 'БРпр1_log_z': 12}

                   count   mean    std    min    max
Стать_ж            121.0  0.620  0.487  0.000  1.000
Стать_ч            121.0  0.380  0.487  0.000  1.000
ступінь_КТ_код      42.0  1.405  0.964  0.000  3.000
тривалість_днів_z  118.0 -0.000  1.000 -2.239  8.174
БРпр1_log_z        109.0  0.000  1.000 -1.926  2.666


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [18]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [19]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [20]:
# УВАГА: цей етап потребує файлу ЛР3_помилковий_аналіз.ipynb, якого немає серед завантажених.
# Заповніть після того, як відкриєте звіт і знайдете десять хибних місць.
ВИПРАВЛЕННЯ = [
    ('1', 'Середнє від номінальної змінної: 1 і 2 — довільні коди «помер» і «жив»',
     'Коди — лише мітки, тож 1,50 не означає «посередині між летальним і сприятливим»: це просто 61 і 60 пацієнтів. Такого проміжного стану не існує.',
     'Рахувати частки: померло 61 (50,4 %), вижило 60 (49,6 %). Для номінальної змінної — мода й частоти.'),
    ('2', 'Різні написання одного значення пораховано як різні категорії',
     'nunique() дає 4, бо «ч»/«чол» і «ж»/«жін» — той самий рівень. Статей у вибірці дві, а «ж» (56) — лише частина з 75.',
     'Звести написання до ч/ж і порахувати частоти: жінок 75 (62 %), чоловіків 46 (38 %).'),
    ('3', 'Середнє від порядкової шкали з алфавітним порядком кодів',
     'sort_index() упорядкував назви за абеткою, а не за тяжкістю («найлегший — важкий»), тож коди довільні; середнє ще й припускає рівні відстані між ступенями.',
     'Очистити назви, задати порядок легкий < середній < важкий < критичний і взяти медіану: «середній» (n = 42).'),
    ('4', 'pd.to_numeric(errors="coerce") мовчки відкинув текстові значення',
     'Діапазони («70-80%») і «до 17%» стали NaN, тому 39,6 % пораховано лише на 27 із 43 рядків, відібраних невипадково.',
     'Спершу звести записи до частки (діапазон → середина). Тоді n = 43: середнє 42,5 %, медіана 35,0 %, σ = 22,0 %.'),
    ('5', 'Неможливе значення SpO2 у даних і висновок зі середнього',
     'Мінімум 0,9 % — помилка введення, вона тягне середнє вниз (82,05 проти 83,29 без неї). Висновок «у більшості» зі середнього не випливає.',
     'Виправити або виключити викид; описувати розподіл: медіана 84 %, частка з SpO2 < 90 % — 75 % (n = 117).'),
    ('6', 'Коефіцієнт варіації для дат (інтервальна шкала)',
     'Нуль дати умовний: від 1970 року CV = 0,595 %, від 1900 — 0,254 %, від 2020 — 15,2 %. Число залежить від точки відліку.',
     'Описувати розкид у днях: від 25.02.2021 до 03.06.2022, розмах 463 дні, σ = 113 днів, або рахувати надходження по місяцях.'),
    ('7', 'Пірсон для двох номінальних змінних і висновок «близько до нуля»',
     'r = 0,177 залежить від кодування (за ч = 1, ж = 0 буде −0,177) і не є мірою лінійного зв’язку; померло 57 % жінок і 39 % чоловіків.',
     'χ² і V Крамера (для 2×2 V = |φ|): χ² = 3,78, p = 0,052, V = 0,18, відношення шансів 2,09 — тенденція, а не «відсутність зв’язку».'),
    ('8', 'Нормалізація за типом даних, а не за шкалою; ділення на нуль',
     'select_dtypes("number") захопив номінальну «помер/жив»; ФЕРИТ2 має єдине значення, max = min, тож 0/0 дало самі NaN і «усі значення в [0, 1]» неправда.',
     'Відбирати стовпці за шкалою, прибрати цільову змінну й сталі стовпці, перевіряти max > min, текстові стовпці спершу очистити.'),
    ('9', 'dropna() за всіма стовпцями; висновок суперечить результату',
     'Було 121 рядок, стало 0: у 69 із 77 стовпців є пропуски, чотири стовпці порожні понад 90 %. «Повністю заповнений набір» — неправда.',
     'Оцінити частку пропусків, прибрати майже порожні стовпці, вибрати потрібні ознаки: Вік, Стать, Л1, Гемогл1, БРпр1, Сечов1 дають 106 рядків.'),
    ('10', 'Витік цільової змінної та кореляції на дуже малому n',
     'Дата розтину є лише у померлих (61), тому «r = 1,00» — це сама відповідь, а не предиктор. ІЛ6-2 на третьому місці пораховано на 6 пацієнтах.',
     'Виключити ознаки, відомі лише після результату; показувати n; кожну пару оцінювати мірою своєї шкали (бісеріальний, Спірмен).'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє від номінальної змінної: 1 і 2 — довільні коди «...,"Коди — лише мітки, тож 1,50 не означає «посередині між л...","Рахувати частки: померло 61 (50,4 %), вижило 60 (49,6 %)..."
1,2,Різні написання одного значення пораховано як різні кате...,"nunique() дає 4, бо «ч»/«чол» і «ж»/«жін» — той самий рі...",Звести написання до ч/ж і порахувати частоти: жінок 75 (...
2,3,Середнє від порядкової шкали з алфавітним порядком кодів,"sort_index() упорядкував назви за абеткою, а не за тяжкі...","Очистити назви, задати порядок легкий < середній < важки..."
3,4,"pd.to_numeric(errors=""coerce"") мовчки відкинув текстові ...","Діапазони («70-80%») і «до 17%» стали NaN, тому 39,6 % п...",Спершу звести записи до частки (діапазон → середина). То...
4,5,Неможливе значення SpO2 у даних і висновок зі середнього,"Мінімум 0,9 % — помилка введення, вона тягне середнє вни...",Виправити або виключити викид; описувати розподіл: медіа...
5,6,Коефіцієнт варіації для дат (інтервальна шкала),"Нуль дати умовний: від 1970 року CV = 0,595 %, від 1900 ...","Описувати розкид у днях: від 25.02.2021 до 03.06.2022, р..."
6,7,Пірсон для двох номінальних змінних і висновок «близько ...,"r = 0,177 залежить від кодування (за ч = 1, ж = 0 буде −...","χ² і V Крамера (для 2×2 V = |φ|): χ² = 3,78, p = 0,052, ..."
7,8,"Нормалізація за типом даних, а не за шкалою; ділення на ...","select_dtypes(""number"") захопив номінальну «помер/жив»; ...","Відбирати стовпці за шкалою, прибрати цільову змінну й с..."
8,9,dropna() за всіма стовпцями; висновок суперечить результату,"Було 121 рядок, стало 0: у 69 із 77 стовпців є пропуски,...","Оцінити частку пропусків, прибрати майже порожні стовпці..."
9,10,Витік цільової змінної та кореляції на дуже малому n,"Дата розтину є лише у померлих (61), тому «r = 1,00» — ц...","Виключити ознаки, відомі лише після результату; показува..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [21]:
# ваш код: три правильні обчислення замість хибних
кт_чист = очищений('КТ').dropna()
print('КТ: n =', len(кт_чист), '| середнє', round(100*кт_чист.mean(), 1),
      '| медіана', round(100*кт_чист.median(), 1), '| σ', round(100*кт_чист.std(), 1))

ст = допустима_статистика('ступінь КТ', 'порядкова')
print('ступінь КТ:', ст['частоти'], '| медіана:', ст['медіана'],
      '| q25:', ст['квантиль 25%'], '| q75:', ст['квантиль 75%'])

т = pd.crosstab(очищений('Стать'), df['помер/жив'].map({1: 'помер', 2: 'жив'}))
χ2_7, p_7, _, _ = stats.chi2_contingency(т, correction=False)
V_7 = np.sqrt(χ2_7 / т.values.sum())
OR_7 = (т.loc['ж', 'помер'] * т.loc['ч', 'жив']) / (т.loc['ж', 'жив'] * т.loc['ч', 'помер'])
print(т); print(f'χ² = {χ2_7:.2f}, p = {p_7:.3f}, V = {V_7:.2f}, OR = {OR_7:.2f}')

КТ: n = 43 | середнє 42.5 | медіана 35.0 | σ 22.0
ступінь КТ: {'легкий': 7, 'середній': 18, 'важкий': 10, 'критичний': 7} | медіана: середній | q25: середній | q75: важкий
помер/жив  жив  помер
Стать                
ж           32     43
ч           28     18
χ² = 3.78, p = 0.052, V = 0.18, OR = 2.09


---
# Етап 7. Висновки та індивідуальне питання

In [22]:
частка_ж = float((очищений('Стать') == 'ж').mean())
n_кт = int(очищений('ступінь КТ').notna().sum())
БРпр1_ряд = очищений('БРпр1').dropna()
топ3_БР = float(БРпр1_ряд.isin([3.4, 5.1, 6.8]).mean())

ВИСНОВКИ = [
    f'Із 121 пацієнта ступінь КТ записано лише для {n_кт}, тож пара «стать x ступінь КТ» '
    f'аналізується на {n} спостереженнях, а решту 79 довелося виключити; будь-який висновок про неї обмежений цією вибіркою.',

    f'Зв’язок статі зі ступенем КТ не виявлено: V Крамера = {V:.2f} (з поправкою на малу вибірку {V_поправлений:.2f}), '
    f'p за перестановками = {p_перестановок:.2f}; очікувані частоти < 5 у {частка_малих:.0%} клітинок, тому χ² тут лише орієнтовний.',

    f'Лише чотири зі восьми стовпців (КТ, Т вище 37,5, Сечов2, БРпр1) допускають середнє й коефіцієнт варіації; '
    f'для телефону, статі та ступеня КТ такі числа безглузді: середній ступінь КТ дорівнює {сер_кт:.1f} або {сер_кт_інакше:.1f} залежно від кодування.',

    f'Значення БРпр1 грубо дискретизовані: {топ3_БР:.0%} вимірювань припадає лише на три числа (3,4; 5,1; 6,8), '
    f'тож тонкі відмінності між пацієнтами тут не розрізнити, хоча шкала формально відношень.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Із 121 пацієнта ступінь КТ записано лише для 42, тож пара «стать x ступінь КТ» аналізується на 42 спостереженнях, а решту 79 довелося виключити; будь-який висновок про неї обмежений цією вибіркою.
2. Зв’язок статі зі ступенем КТ не виявлено: V Крамера = 0.22 (з поправкою на малу вибірку 0.00), p за перестановками = 0.62; очікувані частоти < 5 у 50% клітинок, тому χ² тут лише орієнтовний.
3. Лише чотири зі восьми стовпців (КТ, Т вище 37,5, Сечов2, БРпр1) допускають середнє й коефіцієнт варіації; для телефону, статі та ступеня КТ такі числа безглузді: середній ступінь КТ дорівнює 2.4 або 3.4 залежно від кодування.
4. Значення БРпр1 грубо дискретизовані: 80% вимірювань припадає лише на три числа (3,4; 5,1; 6,8), тож тонкі відмінності між пацієнтами тут не розрізнити, хоча шкала формально відношень.


In [23]:

рез = pd.DataFrame({'Стать': очищений('Стать'),
                    'Результат': df['помер/жив'].map({1: 'помер', 2: 'жив'})}).dropna()
т_рез = pd.crosstab(рез['Стать'], рез['Результат'])
print(т_рез); print()

χ2_р, p_р, dof_р, _ = stats.chi2_contingency(т_рез, correction=False)
n_р = int(т_рез.values.sum())
V_р = float(np.sqrt(χ2_р / n_р))                       # для 2×2 V дорівнює |φ|
OR = (т_рез.loc['ж', 'помер'] * т_рез.loc['ч', 'жив']) / (т_рез.loc['ж', 'жив'] * т_рез.loc['ч', 'помер'])
летальність = (т_рез['помер'] / т_рез.sum(axis=1)).round(3).to_dict()

# Пірсон на кодах 0/1: збігається з φ за модулем, але знак залежить від кодування
x1 = рез['Стать'].map({'ч': 0, 'ж': 1}); x2 = рез['Стать'].map({'ч': 1, 'ж': 0})
y = рез['Результат'].map({'жив': 0, 'помер': 1})
r1 = float(np.corrcoef(x1, y)[0, 1]); r2 = float(np.corrcoef(x2, y)[0, 1])

print(f'χ² = {χ2_р:.2f}, p = {p_р:.3f}, V (=|φ|) = {V_р:.3f}, відношення шансів = {OR:.2f}')
print('Частка померлих:', летальність)
print(f'r Пірсона на кодах: ч=0,ж=1 -> {r1:.3f};  ч=1,ж=0 -> {r2:.3f}')


Результат  жив  помер
Стать                
ж           32     43
ч           28     18



χ² = 3.78, p = 0.052, V (=|φ|) = 0.177, відношення шансів = 2.09
Частка померлих: {'ж': 0.573, 'ч': 0.391}
r Пірсона на кодах: ч=0,ж=1 -> 0.177;  ч=1,ж=0 -> -0.177


In [24]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = f'''
Стать і результат лікування це дві номінальні змінні (ч/ж та помер/жив). Кореляція Пірсона
вимагає, щоб значення були числами на інтервальній шкалі або шкалі відношень: тоді різниці й
порядок мають зміст, а зв’язок описується лінійною залежністю. Коди «ч=0, ж=1» чи «помер=1, жив=2»
це довільні мітки. Це видно на наших даних: з кодами ч=0, ж=1 маємо r = {r1:.3f}, а з ч=1, ж=0
виходить r = {r2:.3f}, тобто знак змінюється від зміни підпису, а не від зміни пацієнтів. Для змінної
з трьома й більше категоріями (наприклад, ступінь тяжкості без порядку) Пірсон не має сенсу
навіть за модулем. Тільки в окремому випадку двох бінарних змінних r на кодах 0/1 за модулем
збігається з коефіцієнтом φ, але його слід називати саме φ, а знак не інтерпретувати.
Замість Пірсона беруть міри для номінальних шкал: критерій χ² незалежності та коефіцієнт Крамера V
(для таблиці 2*2 V = |φ|). У нашій вибірці x^2 = {χ2_р:.2f}, p = {p_р:.3f}, V = {V_р:.3f}; частка померлих
серед чоловіків {летальність['ч']:.0%}, серед жінок {летальність['ж']:.0%}, відношення шансів {OR:.2f}.
Додатково для бінарного результату корисні відношення шансів або ризиків із довірчим інтервалом, а за
малих очікуваних частот точний критерій Фішера.
'''.strip().replace('\n', ' ')
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?

Стать і результат лікування це дві номінальні змінні (ч/ж та помер/жив). Кореляція Пірсона вимагає, щоб значення були числами на інтервальній шкалі або шкалі відношень: тоді різниці й порядок мають зміст, а зв’язок описується лінійною залежністю. Коди «ч=0, ж=1» чи «помер=1, жив=2» це довільні мітки. Це видно на наших даних: з кодами ч=0, ж=1 маємо r = 0.177, а з ч=1, ж=0 виходить r = -0.177, тобто знак змінюється від зміни підпису, а не від зміни пацієнтів. Для змінної з трьома й більше категоріями (наприклад, ступінь тяжкості без порядку) Пірсон не має сенсу навіть за модулем. Тільки в окремому випадку двох бінарних змінних r на кодах 0/1 за модулем збігається з коефіцієнтом φ, але його слід називати саме φ, а знак не інтерпретувати. Замість Пірсона беруть міри для номінальних шкал: критерій χ² незалежності та коефіцієнт Крамера V (для та

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [25]:
# Це поле заповнюєте ви самі й чесно: що саме з цієї роботи писали ви, а що отримали від ШІ.
# Допустимі значення: 'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'синтаксис',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'синтаксис',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'Перевірив чи для кожного стовпця було обчислено тільки те, що дозволяє його шкала, і чи всі обчислення мають зміст. Перевірив, що для кожного стовпця вказано спосіб підготовки. Перевірив, що знайдено 10 помилок у звіті та заповнено їх виправлення. Перевірив, що висновки та відповідь на питання розгорнуті й змістовні.'
ПІДТВЕРДЖУЮ = True    # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [26]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 15 готова до захисту.
Ореньчук Юрій, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.